<div style="background: linear-gradient(135deg, rgb(36, 233, 197) 0%, rgb(16, 109, 202) 30%, rgb(255, 255, 255) 60%);
            padding: 10px; border-radius: 8px; margin: 10px 0; color: black; font-family: Arial, sans-serif; text-align: center; font-size: 24px;">

# **Пайплайн для обучения LLM**
</div>

### Обучение языковой модели: Pretrain с нуля + Post-train SFT дообучение

Проект демонстрирует **два ключевых этапа жизненного цикла языковой модели (LLM)** на практике, от сырых данных до отвечающего на вопросы ассистента.

Цель - реализовать prettain и posttrain этапы обучения LLM.

1. **Pretrain (предобучение с нуля)** - обучение собственной модели архитектуры LLaMA (~132M параметров) на корпусе русской классической литературы. Претрейн является самым ресурсоёмким этапрм обучения LLM. Чтобы полноценно обучить даже небольшую модель (менее 1B), понадобится более 10к GPU-часов на A100, но в данном проекте, на собственной машине чтобы не тратить недели на обучение, но показать отработку основных моментов, в проекте выполняется упрощённая задача.


2. **Post-train SFT (Supervised Fine-Tuning)** - дообучение готовой предобученной модели **Qwen2.5-0.5B** на инструктивном русскоязычном датасете `d0rj/alpaca-cleaned-ru`. Задачей данного этапа будет обучить модель генерировать ответы на русскоязычные вопросы.

По сути делаем из языковой модели ассистента, предобучать собственную модель до уровня, на котором SFT даёт осмысленный результат, на одной видеокарте нереально. Поэтому SFT показан на сильной базовой модели, где эффект дообучения виден за разумное время.

Проект сделан из расчета мощностей железа и наличия одной видеокарты.

<div class='alert alert-info'>

#### **Кстати говоря по железу:**

```terminal
ОС:                  Linux 6.6.87.2-microsoft-standard-WSL2
Окружение:           WSL (Windows Subsystem for Linux)

Процессор:           
Ядер (физ):          12
Ядер (лог):          24
Модель CPU:          13th Gen Intel(R) Core(TM) i7-13700K

ОЗУ всего:           16.6 GB
PyTorch версия:      2.11.0+cu130
CUDA доступна:       True
GPU:                 NVIDIA GeForce RTX 4070
CUDA версия:         13.0
VRAM всего:          12.9 GB
```

<div class='alert alert-info'>

*Так как я столкнулся с проблемой, что не все у нас библиотеки поддерживаются windows, пришлось по изучать работу с wsl.*

**Небольшой гайд для постановки wsl:**

1. PowerShell от администратора

```bash
wsl --install
```

2. Вводим имя пользователя и пароль. Пароль не отображается, так что не стоит по несколько раз нажимать на кнопки, они учитываются. А то я подумал, что клавиатура сломалась и процесс завис

3. WSL иногда не резолвит адреса. Сразу фиксим:

```bash
echo "nameserver 8.8.8.8" | sudo tee /etc/resolv.conf
```

4. Изначально ставится новая версия Ubuntu, у меня была 26.04 - она шла с версией Python 3.14 и PyTorch его не воспринимал.. поэтому ставим более позднюю версию питона 3.11 - с такой работаю на windows ей доверяю.

```bash
sudo add-apt-repository ppa:deadsnakes/ppa -y
sudo apt update
sudo apt install python3.11 python3.11-venv python3.11-dev -y
```

5. Создаем виртуальное окружение

```bash
cd ~
python3.11 -m venv dl_env311
source dl_env311/bin/activate
pip install --upgrade pip
```

6. И ставим библиотеки - с этим было много возни, из-за совместимостей, поэтому вот на этих я и остановился.

```bash
pip install torch==2.5.1+cu121 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install transformers==5.9.0 datasets accelerate==1.13.0 peft==0.19.1 trl==1.4.0 bitsandbytes==0.49.2 jupyter ipykernel tokenizers unsloth
```
*p.s. возможно что-то дополнительно ставил уже в ноутбуке (уже не помню), ну там подскажет терминал, чего не хватит*

7. Я работаю в VScode, поэтому в ней ставлю расширение **WSL**
8. Жмём `Ctrl+shift+p` > `WSL: Connect to WSL using Distro` > `Ubuntu`.
Открываем папку `File - Open Folder` > В строке вводим папку, либо просто новую создаем, я использовал локалку так удобнее для меня, вот мой путь: `/mnt/d/code/ya_neiro/7. Большие языковые модели (LLM).`

9. Открываем терминал `Ctrl + ~` и активируем окружение

```bash
source ~/dl_env311/bin/activate
python -m ipykernel install --user --name=dl_env311 --display-name "Python DL (WSL)" # это мы ядро зарегали в скобках название мое
```

10. Запускаем Jupiter

```bash
jupyter notebook --no-browser --ip=0.0.0.0 --port=8888
```

11. Там (в терминале) выведется информация по ноутбуку, нужно будет копирнуть токен. Выводится два вида, на нужен именно такой - `http://127.0.0.1:8888/?token=...`
12. В VScode **Выбор ядра** > **Существующий сервер Jupyter** > вставляем ссылку
12. Выбери ядро - **Python DL (WSL)** 
13. Запускаем и работаем

*P.S. WSL после перезагрузки у меня теряет DNS. И на повторный запуск используем*
```bash
echo "nameserver 8.8.8.8" | sudo tee /etc/resolv.conf
```

<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(36, 233, 197) 0%, rgb(36, 233, 197) 30%, rgb(255, 255, 255) 60%, rgb(255, 255, 255) 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">БЛОК 1.0. Импорт библиотек</strong>
</div>

In [1]:
import os
import re
import gc
import tempfile
from pathlib import Path

import torch
import unicodedata
import requests

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing

from transformers import (
    PreTrainedTokenizerFast,
    AutoTokenizer,
    AutoModelForCausalLM,
    LlamaConfig,
    LlamaForCausalLM,
    TrainerCallback,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

from datasets import Dataset, load_dataset
from trl import SFTTrainer, SFTConfig

<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(36, 233, 197) 0%, rgb(36, 233, 197) 30%, rgb(255, 255, 255) 60%, rgb(255, 255, 255) 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">БЛОК 1.1. Качаем данные из репозитория</strong>
</div>

Загружаем корпус русской прозы из открытого репозитория **RussianNovels**. Сначала через GitHub API получаем список файлов в папке `corpus`, затем скачиваем каждый `.txt` по прямой ссылке `raw.githubusercontent.com`.

**Почему так выполняется:** - безусловно будут дополнительные проверки и редакции кода и чтобы каждый раз нам не загружать книги и не ожидать, то загрузка пойдет пофайлово с проверкой `if not dest.exists()`, чтобы при повторном запуске ячейки уже скачанные книги не качались заново.

In [2]:
# Скачиваем список файлов из репозитория через GitHub API
REPO_API = "https://api.github.com/repos/JoannaBy/RussianNovels/contents/corpus"
RAW_BASE = "https://raw.githubusercontent.com/JoannaBy/RussianNovels/master/corpus"
DATA_DIR = Path("./russian_novels")
DATA_DIR.mkdir(exist_ok=True)

response = requests.get(REPO_API)
files = response.json()

txt_files = [f for f in files if f["name"].endswith(".txt")]
print(f"Найдено файлов: {len(txt_files)}")

for f in txt_files:
    dest = DATA_DIR / f["name"]
    if not dest.exists():
        raw = requests.get(f"{RAW_BASE}/{f['name']}")
        dest.write_bytes(raw.content)
        print(f"Скачал: {f['name']}")
    else:
        print(f"Уже есть: {f['name']}")

print("\Загрузка завершена!")

Найдено файлов: 108
Уже есть: Bulgakov_BelayaGvardiya.txt
Уже есть: Bulgakov_Diavoliada.txt
Уже есть: Bulgakov_Master.txt
Уже есть: Bulgakov_RokovyeYaytsa.txt
Уже есть: Bulgakov_TeatralnyjRoman.txt
Уже есть: Bulgakov_ZapiskiYonogoVracha.txt
Уже есть: Chekhov_Dama.txt
Уже есть: Chekhov_Dyadya.txt
Уже есть: Dostoyevski_Biesy.txt
Уже есть: Dostoyevski_Idiot.txt
Уже есть: Dostoyevsky_BednyeLyudi.txt
Уже есть: Dostoyevsky_Karamazow1.txt
Уже есть: Dostoyevsky_Karamazow2.txt
Уже есть: Dostoyevsky_Karamazow3.txt
Уже есть: Dostoyevsky_Karamazow4.txt
Уже есть: Dostoyevsky_PrestupleniyeINakazaniye.txt
Уже есть: Durova_GodZyznijWPeterburge.txt
Уже есть: Durova_Gudishki.txt
Уже есть: Durova_IgraSudby.txt
Уже есть: Durova_Pavilion.txt
Уже есть: Durova_SernyjKljuch.txt
Уже есть: Durova_Ugol.txt
Уже есть: Durova_Yurchak.txt
Уже есть: Gan_Ideal.txt
Уже есть: Gan_SudSveta.txt
Уже есть: Gippius_ChertovayaKukla.txt
Уже есть: Gippius_Roman-Tzarevich.txt
Уже есть: Gogol_Mertvye.txt
Уже есть: Gogol_Starosvye

<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(36, 233, 197) 0%, rgb(36, 233, 197) 30%, rgb(255, 255, 255) 60%, rgb(255, 255, 255) 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">БЛОК 1.2. Загружаем файлы и чистим тексты</strong>
</div>

В данном разделе мы прочитаем скаченные файлы и проведем небольшую чистку, для приведения к подобающему виду для моделей. 

Что мы сделаем - переберем кодировки текстов, т.к. копаться и смотреть что и какой не хочется, это временно затратно. А потом нормализуем Unicode и построчно отбросим строки без кириллицы и строки, где кириллицы менее 50%, это по сути наши номера страниц, какие-то разметки и т.п. Работаем над пунктуацией и лишних пробелом и убираем короткие строки.

In [3]:
def load_text(path):
    """Функция загрузки текста из файла с попыткой разных кодировок"""       
    for enc in ["utf-8", "windows-1251", "latin-1"]:
        try:
            return Path(path).read_text(encoding=enc)
        except UnicodeDecodeError:
            continue
    return ""

def clean_text(text):
    """Очистка текста от мусора, нормализация и фильтрация по кириллице

    Args:
        text (_type_): текст для очистки

    Returns:
        _type_: очищенный текст
    """    
   
    # Нормализация unicode
    text = unicodedata.normalize("NFC", text)
    
    # Убираем строки без кириллицы (оставляем только строки где есть кириллица)
    lines = text.split("\n")
    clean_lines = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        # Проверяем есть ли кириллица в строке
        has_cyrillic = bool(re.search(r'[а-яёА-ЯЁ]', line))
        if not has_cyrillic:
            continue
        # Убираем строки где кириллица меньше 50% символов (латиница, цифры и т.д.)
        cyrillic_count = len(re.findall(r'[а-яёА-ЯЁ]', line))
        total_alpha = len(re.findall(r'[a-zA-Zа-яёА-ЯЁ]', line))
        if total_alpha > 0 and cyrillic_count / total_alpha < 0.5:
            continue
        clean_lines.append(line)
    
    text = " ".join(clean_lines)
    
    # Убираем повторяющуюся пунктуацию
    text = re.sub(r'([.,!?;:]){2,}', r'\1', text)
    text = re.sub(r'\.{3,}', '...', text)
    
    # Убираем лишние пробелы
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()

# Загружаем все тексты
all_texts = []
for txt_file in sorted(DATA_DIR.glob("*.txt")):
    raw = load_text(txt_file)
    cleaned = clean_text(raw)
    if len(cleaned) > 1000:  # Задаём минимальный порог для объёма текста, у меня 1000
        all_texts.append(cleaned)
        print(f"  {txt_file.name}: {len(cleaned):,} символов")

print(f"\nВсего текстов: {len(all_texts)}")
print(f"Общий объём: {sum(len(t) for t in all_texts):,} символов")

  Bulgakov_BelayaGvardiya.txt: 472,576 символов
  Bulgakov_Diavoliada.txt: 66,380 символов
  Bulgakov_Master.txt: 750,185 символов
  Bulgakov_RokovyeYaytsa.txt: 137,572 символов
  Bulgakov_TeatralnyjRoman.txt: 271,451 символов
  Bulgakov_ZapiskiYonogoVracha.txt: 211,116 символов
  Chekhov_Dama.txt: 326,186 символов
  Chekhov_Dyadya.txt: 78,985 символов
  Dostoyevski_Biesy.txt: 1,252,721 символов
  Dostoyevski_Idiot.txt: 1,304,270 символов
  Dostoyevsky_BednyeLyudi.txt: 252,313 символов
  Dostoyevsky_Karamazow1.txt: 374,503 символов
  Dostoyevsky_Karamazow2.txt: 394,320 символов
  Dostoyevsky_Karamazow3.txt: 426,720 символов
  Dostoyevsky_Karamazow4.txt: 621,460 символов
  Dostoyevsky_PrestupleniyeINakazaniye.txt: 1,076,848 символов
  Durova_GodZyznijWPeterburge.txt: 114,818 символов
  Durova_Gudishki.txt: 479,423 символов
  Durova_IgraSudby.txt: 125,345 символов
  Durova_Pavilion.txt: 216,125 символов
  Durova_SernyjKljuch.txt: 56,370 символов
  Durova_Ugol.txt: 185,169 символов
  Duro

В целом неплохой объем символов для pretrain. Но пробежавшись по текстам глазами, я обнаружил возможный дубликат:

- Vovchok_Zapiski prichyotnika.txt: 615,615 символов
- Vovchok_ZapiskiPrichyotnika.txt: 615,615 символов

Чем больше тем лучше, но пожалуй убиремдубликаты текстов по длине и первым символам, может я где ошибся и не заметил

In [4]:
seen = set()
unique_texts = []
for t in all_texts:
    key = (len(t), t[:100])
    if key not in seen:
        seen.add(key)
        unique_texts.append(t)
all_texts = unique_texts
print(f"Текстов после чистки: {len(all_texts)}")

Текстов после чистки: 107


Вроде норм, едемдальше

<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(36, 233, 197) 0%, rgb(36, 233, 197) 30%, rgb(255, 255, 255) 60%, rgb(255, 255, 255) 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">БЛОК 1.3. Препроцессинг: разбивка на предложения и дедупликация</strong>
</div>

Тут мы разобьем предложения по знакам конца предложений и уберем дубликаты, а также уберем повторяющиеся фрагменты.

In [5]:
def split_into_sentences(text):
    """Ращбиваем текст на предложения

    Args:
        text (_type_): Текст для разбивки

    Returns:
        _type_: текст разбитый на предложения
    """    
    # Разбиваем по точке, восклицательному и вопросительному знаку
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]
    return sentences

# Собираем все предложения
all_sentences = []
for text in all_texts:
    sentences = split_into_sentences(text)
    all_sentences.extend(sentences)

print(f"Предложений до чистки: {len(all_sentences):,}")

# убираем точные дубликаты
all_sentences = list(dict.fromkeys(all_sentences))

print(f"Предложений после чистки: {len(all_sentences):,}")

# Выведим парочку примеров
print(f"\nПримеры предложений:")
for s in all_sentences[:5]:
    print(f"  {s[:100]}")

Предложений до чистки: 441,384
Предложений после чистки: 438,032

Примеры предложений:
  Посвящается Любови Евгеньевне Белозерской Пошел мелкий снег и вдруг повалил хлопьями.
  Ветер завыл; сделалась метель.
  В одно мгновение темное небо смешалось с снежным морем.
  - Ну, барин, - закричал ямщик, - беда: буран!
  "Капитанская дочка" И судимы были мертвые по написанному в книгах сообразно с делами своими.


Убрали небольшой процент дубликатов -тоже хорошо, можнобить на чанки

<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(36, 233, 197) 0%, rgb(36, 233, 197) 30%, rgb(255, 255, 255) 60%, rgb(255, 255, 255) 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">БЛОК 1.4. Разбивка на чанки</strong>
</div>

Тут мы разобьем на чанки - кароче говоря в блоки, опредленного фиксированного размера. При нашей заданной длинной контекста 512 токенов в виде экземпляра класса `transformers.Dataset` Размер чанка ставим 1800.

In [6]:
CONTEXT_LENGTH = 512  # длина контекста 
CHUNK_CHAR_SIZE = 1800  # размер чанка в символах

def create_chunks(sentences, chunk_size=CHUNK_CHAR_SIZE):
    """Собираем предложения в чанки

    Args:
        sentences (_type_): список предложений для объединения в чанки
        chunk_size (_type_, optional): Размер чанка в символах. Defaults to 1800

    Returns:
        _type_: список чанков
    """    
    chunks = []
    current_chunk = []
    current_len = 0
    
    for sentence in sentences:
        if current_len + len(sentence) > chunk_size and current_chunk:
            chunk_text = " ".join(current_chunk)
            chunks.append(chunk_text)
            current_chunk = [sentence]
            current_len = len(sentence)
        else:
            current_chunk.append(sentence)
            current_len += len(sentence)
    
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    return chunks

chunks = create_chunks(all_sentences)
print(f"Чанков: {len(chunks):,}")
print(f"Средняя длина чанка: {sum(len(c) for c in chunks)//len(chunks):,} символов")
print(f"\nПример чанка:")
print(chunks[0][:300])

Чанков: 24,024
Средняя длина чанка: 1,733 символов

Пример чанка:
Посвящается Любови Евгеньевне Белозерской Пошел мелкий снег и вдруг повалил хлопьями. Ветер завыл; сделалась метель. В одно мгновение темное небо смешалось с снежным морем. - Ну, барин, - закричал ямщик, - беда: буран! "Капитанская дочка" И судимы были мертвые по написанному в книгах сообразно с дел


Ну в целом при средней длине чанка 1733 у нас чанки заполняются почти безпотерь, и вписываются в длину контекста токенов 512, тут останется место для [BOS] и [EOS] токенов.

<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(36, 233, 197) 0%, rgb(36, 233, 197) 30%, rgb(255, 255, 255) 60%, rgb(255, 255, 255) 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">БЛОК 1.5. Обучение токенизатора</strong>
</div>

Тут мы зададим словарь около 3к токенов. Почему такой? Потому что у нас есть ограничения по железу и было бы разумно взять для небольшой модели такое количество, ну и в задании тоже так сказано)) 

*В рассматриваемых данных язык намного менее разнообразен, чем в совокупных данных, поэтому крупные токенизаторы от реальных LLM могут не подойти.* 

Меньше эмбеддингов. Ну и соответственно обучим собственны токенизатор.

In [9]:
VOCAB_SIZE = 3000

# Сохраняем чанки во временный файл для обучения токенизатора
with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', suffix='.txt', delete=False) as f:
    for chunk in chunks:
        f.write(chunk + "\n")
    temp_file = f.name

print(f"Временный файл: {temp_file}")

# Создаём и обучаем BPE токенизатор
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["[UNK]", "[BOS]", "[EOS]", "[PAD]"],
    min_frequency=2,
    show_progress=True,
)

tokenizer.train([temp_file], trainer)

# Добавляем постпроцессинг - BOS в начало и EOS в конец
tokenizer.post_processor = TemplateProcessing(
    single="[BOS] $A [EOS]",
    special_tokens=[
        ("[BOS]", tokenizer.token_to_id("[BOS]")),
        ("[EOS]", tokenizer.token_to_id("[EOS]")),
    ],
)

# Сохраняем токенизатор
tokenizer.save("./my_tokenizer.json")

print(f"ID токенов: BOS={tokenizer.token_to_id('[BOS]')}, EOS={tokenizer.token_to_id('[EOS]')}, PAD={tokenizer.token_to_id('[PAD]')}")

# Тест
test = "Война не любезность, а самое гадкое дело в жизни."
encoded = tokenizer.encode(test)
print(f"\nТест токенизатора:")
print(f"Текст: {test}")
print(f"Токены: {encoded.tokens[:15]}")
print(f"IDs: {encoded.ids[:15]}")

Временный файл: /tmp/tmp6gv0lkmu.txt



ID токенов: BOS=1, EOS=2, PAD=3

Тест токенизатора:
Текст: Война не любезность, а самое гадкое дело в жизни.
Токены: ['[BOS]', 'Во', 'й', 'на', 'не', 'любез', 'ность', ',', 'а', 'самое', 'га', 'д', 'кое', 'дело', 'в']
IDs: [1, 750, 166, 214, 216, 2824, 2966, 18, 157, 2054, 314, 161, 560, 772, 159]


В целом редкие слова бьются на части - это является нормальным поведением BPE при нашем небольшом словаре 3000.Частые к примеру - не, а, самое, дело - остаютсяцелыми.
Словарь конечно маловат, из-за чего возможно редкие слова будут дробиться сильно, но двигаемся по изначальному плану задания.

<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(36, 233, 197) 0%, rgb(36, 233, 197) 30%, rgb(255, 255, 255) 60%, rgb(255, 255, 255) 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">БЛОК 1.6. Подготока датасета к Pretrain, инициализация модели и создание TrainerCallback</strong>
</div>

Обернем обученный токенизатор в формат понятный transformers, токенизируем чанки и сформируем датасет

In [10]:
# Оборачиваем в HF токенизатор
hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="./my_tokenizer.json",
    bos_token="[BOS]",
    eos_token="[EOS]",
    unk_token="[UNK]",
    pad_token="[PAD]",
)

def tokenize_function(examples):
    """Токенизируем чанки   

    Args:
        examples (_type_): примеры для токенизации

    Returns:
        _type_: токенизированные примеры
    """    
    return hf_tokenizer(
        examples["text"],
        truncation=True,
        max_length=CONTEXT_LENGTH,
        padding="max_length",
    )

# Создаём датасет
raw_dataset = Dataset.from_dict({"text": chunks})
print(f"Размер датасета: {len(raw_dataset)}")

# Токенизируем
tokenized_dataset = raw_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Токенизация",
)

# Добавляем labels
def add_labels(examples):
    examples["labels"] = examples["input_ids"].copy()
    return examples

tokenized_dataset = tokenized_dataset.map(add_labels, batched=True, desc="Добавление labels")

# Разбиваем на train/val
split = tokenized_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
val_dataset = split["test"]

print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")
print(f"Пример батча: {list(tokenized_dataset.features.keys())}")

Размер датасета: 24024


Токенизация:   0%|          | 0/24024 [00:00<?, ? examples/s]

Добавление labels:   0%|          | 0/24024 [00:00<?, ? examples/s]

Train: 22822
Val: 1202
Пример батча: ['input_ids', 'attention_mask', 'labels']


Создадим модель **LLaMA** со случайной инициализацией весов

Конфигурация и обоснование выбора:
- `hidden_size=1024`, `num_hidden_layers=16`, `num_attention_heads=16` - с учетом видяхи 12Gb надеюсь потянем предложенный пример.
- `num_key_value_heads=8` - 16 голов на 8 групп K/V.
- `rope_theta=10000` - RoPE-позиционные эмбеддинги (дефолт для LLaMA).
- `hidden_act="silu"` - функция активации, как в оригинальной LLaMA.

In [11]:
config = LlamaConfig(
    vocab_size=hf_tokenizer.vocab_size,
    hidden_size=1024,
    intermediate_size=1536,
    num_hidden_layers=16,
    num_attention_heads=16,
    num_key_value_heads=8,
    max_position_embeddings=CONTEXT_LENGTH,
    bos_token_id=hf_tokenizer.bos_token_id,
    eos_token_id=hf_tokenizer.eos_token_id,
    pad_token_id=hf_tokenizer.pad_token_id,
    rope_theta=10000.0,
    hidden_act="silu",
)

model = LlamaForCausalLM(config)

# Подсчёт параметров
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Всего параметров: {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,}")
print(f"Примерно: {total_params/1e6:.1f}M")

Всего параметров: 132,006,912
Обучаемых параметров: 132,006,912
Примерно: 132.0M


Создаём `TrainerCallback`, который **каждые 500 шагов** генерирует продолжения нескольких тестовых промптов. И взглянем картину прогресса, как модель переходит к тексту. 

In [13]:
# Промты с проекта
test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
]

class GenerationCallback(TrainerCallback):
    """Коллбэк для генерации текста на каждой эпохе

    Args:
        TrainerCallback (_type_): родительский класс для коллбэков в HF Trainer
    """    
    
    def __init__(self, tokenizer, prompts, every_n_steps=500):
        """Иниты

        Args:
            tokenizer (_type_): токенизатор для подготовки промптов
            prompts (_type_): список промптов для генерации
            every_n_steps (int, optional): интервал шагов для генерации. Defaults to 500.
        """        
        self.tokenizer = tokenizer
        self.prompts = prompts
        self.every_n_steps = every_n_steps
    
    def on_step_end(self, args, state, control, model=None, **kwargs):
        """ Генерируем текст по промптам каждый интервал шагов

        Args:
            args (_type_): аргументы тренера
            state (_type_): состояние тренера, содержит информацию о текущем шаге
            control (_type_): контрольный объект
            model (_type_, optional): модель для генерации. Defaults to None.
        """        
        if state.global_step % self.every_n_steps != 0 or state.global_step == 0:
            return
        
        print(f"Шаг {state.global_step} - примеры генерации:")
        
        model.eval()
        device = next(model.parameters()).device
        
        with torch.no_grad():
            for prompt in self.prompts[:3]:  # Показываем первые 3
                inputs = self.tokenizer(
                    prompt,
                    return_tensors="pt",
                    truncation=True,
                    max_length=50
                ).to(device)
                
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=50,
                    do_sample=True,
                    temperature=0.8,
                    top_p=0.9,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
                
                generated = self.tokenizer.decode(
                    outputs[0][inputs["input_ids"].shape[1]:],
                    skip_special_tokens=True
                )
                print(f"\nПромпт: {prompt}")
                print(f"Генерация: {generated}")
        
        model.train()

generation_callback = GenerationCallback(hf_tokenizer, test_prompts, every_n_steps=500)

Ну теперь можно перейти непосредственно к обучению

<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(36, 233, 197) 0%, rgb(36, 233, 197) 30%, rgb(255, 255, 255) 60%, rgb(255, 255, 255) 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">БЛОК 1.6. Обучение Pretrain</strong>
</div>

Сразу оговорка в задании сказано *Используйте подходящий batch_size — в диапазоне 64—128.*
**Но я снижу всё это дело в связи с моей RTX4070 - 12GB в сравнении с NVIDIA Tesla T4 - 16 GB**

Итого гиперпараметры:

- `gradient_accumulation_steps=8` при `batch_size=8` должен дать эффективный батч 64. Накопление градиента позволит имитировать большой батч на более слабой видеокарте.
- `fp16=True` - обучение в смешанной точности: быстрее и экономит память.
- `learning_rate=3e-4` с `lr_scheduler_type="cosine"` и `warmup_ratio=0.05` - плавный разогрев, затем косинусное затухание.
- `weight_decay=0.1` - делаем регуляризацию, чтобы нам не переобучиться.
- `DataCollatorForLanguageModeling(mlm=False)` - режим **CLM** (предсказание следующего токена).
- `dataloader_num_workers=0` — для совместимости с WSL.

In [14]:
training_args = TrainingArguments(
    output_dir="./pretrain_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,
    weight_decay=0.1,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=hf_tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[generation_callback],
)

print(f"Шагов на эпоху: {len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Шагов на эпоху: 356


Step,Training Loss,Validation Loss
500,3.824433,3.830149
1000,3.340120,3.584293
1071,3.340120,3.583009


Шаг 500 - примеры генерации:

Промпт: Все мысли, которые имеют огромные последствия
Генерация: же , когда она была не прият на , и потому , что они с рав ни вали , и что , по сло вам их , как будто я бы , чтобы не за думы ваться . На вопрос , что я еще не могу понять , что мы

Промпт: Сила войска зависит от его духа
Генерация: . По дож дите , пока ме ст ные ! А не которые , на чи ная - то ! - И так , - не решительно сказала Наталь я . - Да разве у вас на ря ди ли ? - Я не хочу , - сказал Три ро

Промпт: Мысль о том, что он принес страдания
Генерация: нова того , и не мог бы было бы с держа ть , но что если бы не было бы ни у меня , ни только для того , чтобы не со держа ть с мы сла . Я про жи ву к этой любви . Я не понимаю ,
Шаг 1000 - примеры генерации:

Промпт: Все мысли, которые имеют огромные последствия
Генерация: з ей , все вы шло , и вы могли бы быть , чтобы вы про па д ались , чтобы вы з вать его на вы ста вку и не да вать себе у слу г . Но вы бу дете люби ть эту че сть . Да не

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1071, training_loss=4.075895863309516, metrics={'train_runtime': 1396.512, 'train_samples_per_second': 49.026, 'train_steps_per_second': 0.767, 'total_flos': 2.7118564408295424e+16, 'train_loss': 4.075895863309516, 'epoch': 3.0})

- видим, что Loss  у нас снижается, можно сказать, что модель учится.
- Train и Vall близки  - знаков переобучения сильного нет
- Мы видим, что со следующим шагов из 500 на 1000 - фрагменты появляются связные. Но тем неменее у нас слова разбитые получаются, это видимо плата за нашу эконмию ресурсов и времени.
- Нужно учитывать, что мы и с батчем поиграли и словарь у нас не такой уж и большой в 3000 токенов. 
- Думаю, что с увеличением словаря и добавлениям ещё 2 - 3 эпох можно и улучшить результаты. Надеюсь, что в рамках проекта, данного результата будет достаточно.

Сохраним модель и токенизатор

In [15]:
model.save_pretrained("./pretrained_model")
hf_tokenizer.save_pretrained("./pretrained_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./pretrained_model/tokenizer_config.json',
 './pretrained_model/tokenizer.json')

<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(36, 233, 197) 0%, rgb(36, 233, 197) 30%, rgb(255, 255, 255) 60%, rgb(255, 255, 255) 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">БЛОК 1.7. Тестовая генерация на промтах</strong>
</div>

In [16]:
model.eval()
device = next(model.parameters()).device

for i, prompt in enumerate(test_prompts, 1):
    inputs = hf_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=50
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=hf_tokenizer.pad_token_id,
            eos_token_id=hf_tokenizer.eos_token_id,
        )
    
    generated = hf_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    
    print(f"\nПромпт {i}: {prompt}")
    print(f"Ответ: {generated}")


Промпт 1: Все мысли, которые имеют огромные последствия
Ответ: з ну , которые можно было раз ру шить и со ставля ть , и те , которые так много на помина ли ее . И это была не от ло жно сть и пре ступ ная улыб ка , но за то тем ная , что уже с пле т ни чая . Она была только тогда в доме , когда он за ходил ко мне на встре чу , и только у по мя нул о том ,

Промпт 2: Сила войска зависит от его духа
Ответ: хо вого о зе ра и при ве дет к себе в а с ф аль т и к само му дому , к не сколь ким раз вра ту и пре зи ра ю . - А я знаю , что он не хочет по говорить о ва ших дела х , - сказал м - сь е Пьер , улыбаясь , - в ка ком случае ты , может быть , так уж поз дно ? - В

Промпт 3: Мысль о том, что он принес страдания
Ответ: стей шей части ю , а если он не будет у стро ить в жизни его и в голову , то мы до жи да ем ся , что это - то и есть . О тец мой был бы еще более на пу га н , чем бы он ни хотел сделать , если бы он мог взя ть бы под руки на пол ную у трен нюю вы году . Но теперь он знал , 

Ну что тут можнос сказать, в качестве улучшений можем увеличить словарь, использовать другой алгоритм токенизации или же использовать готовый токенизатор - но это не совсем подойдет для нашего проекта. 

Модель выучила язык: морфологию, согласование, пунктуацию, литературный регистр. При этом она,не отвечает на вопросы и не следует инструкциям, а просто продолжает подобный текст.

Для повышения качества больше данных нужно и улучить параметры.

<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(16, 109, 202) 0%, rgb(16, 109, 202) 30%, rgb(255, 255, 255) 60%, rgb(255, 255, 255) 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">БЛОК 2.1. Post-train SFT</strong>
</div>

Тут мы будем создавать модель ассистента отвечающего на вопросы из готовой предобученной модели **Qwen2.5-0.5B** и дообучаем её методом **SFT** на инструктивном русскоязычном датасете `d0rj/alpaca-cleaned-ru`.

Но для начала перед загрузкой модели, освободим нашу видеопамять и удалим объекты model и train, т.к. на нашей видяхе не поместятся две модели одновременно, и могут возникать ошибки с памятью

In [17]:
try:
    del model
    del trainer
except:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

print("Память очищена")

Память очищена


<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(16, 109, 202) 0%, rgb(16, 109, 202) 30%, rgb(255, 255, 255) 60%, rgb(255, 255, 255) 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">БЛОК 2.2. Загрузка Qwen2.5-0.5B</strong>
</div>

Загружаем токенизатор и веса Qwen2.5-0.5B.

Прошу обратить внимание, что использован `torch_dtype=torch.float32`, так как при обучении в `float16` выкидывалось NaN. 

In [18]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

print(f"Модель - {MODEL_NAME}")

sft_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

# Qwen2.5 не имеет pad_token - используем eos
if sft_tokenizer.pad_token is None:
    sft_tokenizer.pad_token = sft_tokenizer.eos_token


sft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,   # было float16
    device_map="auto",
    trust_remote_code=True,
)


total = sum(p.numel() for p in sft_model.parameters())
print(f"\nМодель загружена: {total/1e6:.1f}M параметров")
print(f"Токенизатор: vocab_size={sft_tokenizer.vocab_size}")

Модель - Qwen/Qwen2.5-0.5B


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Модель загружена: 494.0M параметров
Токенизатор: vocab_size=151643


Отлично модель загружена. 

Выполним генерацию до обучения. Взглянем, как базовая модель отвечает на простые вопросы без SFT - будет наш сравнительный эталон или отправная точка

In [19]:
questions_rus = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
]

def generate_answer(model, tokenizer, question, max_new_tokens=150):
    """Генерация ответа на вопрос

    Args:
        model (_type_): модель
        tokenizer (_type_): токенизатор
        question (_type_): вопрос
        max_new_tokens (int, optional): максимальное количество новых токенов. Defaults to 150.

    Returns:
        _type_: ответ
    """    
    inputs = tokenizer(question, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )


print("=== Base Model (Before SFT) Output ===")

for i, q in enumerate(questions_rus, 1):
    answer = generate_answer(sft_model, sft_tokenizer, q)
    print(f"\nModel Input {i}:\n{q}")
    print(f"Model Output {i}:\n{answer}")

print("\n" + "=" * 70)

=== Base Model (Before SFT) Output ===

Model Input 1:
сколько планет в нашей солнечной системе?
Model Output 1:
 - Метки: Новое Земное Солнце
Самый большой планета в нашей солнечной системе - планета Марс, находящаяся в 180-м знаке Стрельца. Внутри планеты есть палитра, состоящая из белых и серых камней, которые образуют панораму планеты. Марс обладает самым низким уровнем освещения среди планет и обладает самым низким уровнем температуры, а также самым низким уровнем температурной длины. Также самая большая план

Model Input 2:
расскажи стих
Model Output 2:
отворение с драмой поэт

Слушайте стихотворение с драмой поэта, которое рассказывает о мечтаниях, которые могли бы вызвать в будущем.

Model Input 3:
когда собирать крыжовник?
Model Output 3:
 | Попытка дождаться появления российских крыжовников | Крыжовник | Новости
Крыжовник: когда собирать крыжовник?
Скоро у нас будут крыжовники, но пока мы еще не знаем, когда их будут сборять, а так же, что будет их сборка. Вчера я читал в одн

Модель отвечает по русски - это уже не плохо, но есть и ошибочки, к примеру окончания не корректные `Самый большой планета`, ну и по факту не `Марс` - самая большая планета в нашей солнечной системе, да и вопрос о другом...

Model Output 2: `отворение с драмой поэт` - обрезаны слова

Ну и так далее.. тут можно анализировать и разбирать


<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(16, 109, 202) 0%, rgb(16, 109, 202) 30%, rgb(255, 255, 255) 60%, rgb(255, 255, 255) 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">БЛОК 2.3. Подготовка датасета d0rj/alpaca-cleaned-ru</strong>
</div>

Что такое вообще - d0rj/alpaca-cleaned-ru - это открытый датасет это открытый датасет, который представляет собой набор пар «инструкция - ответ» на современном русском языке. Он содержит около 50 000 примеров. Привет из - https://habr.com/ru/articles/971266/

Мы будем загружать и форматировать наш инструктивный датасет.

In [20]:
alpaca_dataset = load_dataset("d0rj/alpaca-cleaned-ru", split="train")
print(f"Размер датасета: {len(alpaca_dataset)}")
print(f"Колонки: {alpaca_dataset.column_names}")
print(f"Пример:")
print(alpaca_dataset[0])

Размер датасета: 51760
Колонки: ['input', 'instruction', 'output']
Пример:
{'input': '', 'instruction': 'Дайте три совета, как оставаться здоровым.', 'output': '1. Соблюдайте сбалансированную и питательную диету. Убедитесь, что в ваш рацион входят разнообразные фрукты и овощи, нежирный белок, цельнозерновые продукты и полезные жиры. Это помогает обеспечить ваш организм необходимыми питательными веществами для оптимального функционирования и может помочь предотвратить хронические заболевания.\n\n2. Занимайтесь регулярной физической активностью. Упражнения имеют решающее значение для поддержания крепких костей, мышц и здоровья сердечно-сосудистой системы. Старайтесь уделять не менее 150 минут умеренным аэробным упражнениям или 75 минут интенсивным упражнениям каждую неделю.\n\n3. Высыпайтесь. Достаточное количество качественного сна имеет решающее значение для физического и психического благополучия. Он помогает регулировать настроение, улучшать когнитивные функции и поддерживает здоровы

In [ ]:
def format_dialog(example):
    """Форматируем  в диалоговый формат для обучения

    Args:
        example (_type_): пример из датасета

    Returns:
        _type_: отформатированный пример
    """    
    messages = []
    
    # System
    system_text = example.get("input", "").strip()
    if system_text:
        messages.append({"role": "system", "content": system_text})
    else:
        messages.append({"role": "system", "content": "Ты полезный русскоязычный ассистент."})
    
    # User
    messages.append({"role": "user", "content": example["instruction"]})
    
    # Assistant
    messages.append({"role": "assistant", "content": example["output"]})
    
    text = sft_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

# Форматируем
formatted_dataset = alpaca_dataset.map(
    format_dialog,
    remove_columns=alpaca_dataset.column_names,
    desc="Форматирование диалогов",
)

# Убираем слишком длинные примеры
MAX_LEN = 1024
def filter_long(example):
    tokens = sft_tokenizer(example["text"], truncation=False)
    return len(tokens["input_ids"]) <= MAX_LEN

formatted_dataset = formatted_dataset.filter(filter_long, desc="Фильтрация длинных")

# Train/val split
split = formatted_dataset.train_test_split(test_size=0.05, seed=42)
sft_train = split["train"]
sft_val = split["test"]

# Тут я умышленно ограничил количество примеров, чтобы ускорить обучение, иначе встряну часов на 6
sft_train = sft_train.select(range(10000))
sft_val = sft_val.select(range(500))



print(f"\nTrain: {len(sft_train)}")
print(f"Val: {len(sft_val)}")
print(f"\nПример отформатированного диалога:")
print(sft_train[0]["text"][:500])


Train: 10000
Val: 500

Пример отформатированного диалога:
<|im_start|>system
Ты полезный русскоязычный ассистент.<|im_end|>
<|im_start|>user
Выведите логарифм числа 100 по основанию 10.<|im_end|>
<|im_start|>assistant
Логарифм числа 100 по основанию 10 равен 2.<|im_end|>



<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(16, 109, 202) 0%, rgb(16, 109, 202) 30%, rgb(255, 255, 255) 60%, rgb(255, 255, 255) 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">БЛОК 2.4. SFT-обучение через TRL</strong>
</div>

Собственно говоря дообучаем модель с помощью trl SFTTrainer

In [23]:
sft_config = SFTConfig(
    output_dir="./sft_checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=False,
    fp16=False,
    max_grad_norm=1.0,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=400,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    max_length=512,
    dataset_text_field="text",
    dataloader_num_workers=0,
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_config,
    train_dataset=sft_train,
    eval_dataset=sft_val,
    processing_class=sft_tokenizer,
)

print("Начинаем SFT обучение...")
sft_trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

...

Начинаем SFT обучение...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,1.329458,1.302216,1.351332,1677856.000000,0.697627
313,1.303760,1.282189,1.334656,2616004.000000,0.702038


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=313, training_loss=1.387745653097622, metrics={'train_runtime': 3080.0885, 'train_samples_per_second': 3.247, 'train_steps_per_second': 0.102, 'total_flos': 7686277117009920.0, 'train_loss': 1.387745653097622, 'epoch': 1.0})

Траейн и лосс близки. После обучения видно, как я схалтурил с количеством примеров и `eval_steps=200, save_steps=400,` По хорошему лучше поставить `eval_steps=100, save_steps=200`... Accuracy у нас растет - что хорошо, энтропия снижается - модель более уверенна становится в ответах.

Сохраним полученную модель и проведем тест

In [30]:
sft_model.save_pretrained("./sft_model")
sft_tokenizer.save_pretrained("./sft_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./sft_model/tokenizer_config.json',
 './sft_model/chat_template.jinja',
 './sft_model/tokenizer.json')

<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(16, 109, 202) 0%, rgb(16, 109, 202) 30%, rgb(255, 255, 255) 60%, rgb(255, 255, 255) 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">БЛОК 2.5. Проверка на адекватность генерации</strong>
</div>

Загружаем сохранённую SFT-модель и прогоняем те же самые вопросы, и сравниваем сравнение до/после. С теми же настройками модели.

In [ ]:
# Загружаем сохранённую SFT модель
sft_tokenizer = AutoTokenizer.from_pretrained("./sft_model")
sft_model = AutoModelForCausalLM.from_pretrained(
    "./sft_model",
    dtype=torch.float32,
    device_map="auto",
)


def generate_answer(model, tokenizer, question, max_new_tokens=150):
    messages = [
        {"role": "system", "content": "Ты полезный русскоязычный ассистент."},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )


questions_rus = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
]


print("=== Base Model (After SFT) Output ===")


for i, q in enumerate(questions_rus, 1):
    answer = generate_answer(sft_model, sft_tokenizer, q)
    print(f"\nModel Input {i}:\n{q}")
    print(f"Model Output {i}:\n{answer}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

=== Base Model (After SFT) Output ===

Model Input 1:
сколько планет в нашей солнечной системе?
Model Output 1:
Согласно данным Международного астрономического общества (IAU), в нашей солнечной системе есть 8 планеты.


Model Input 2:
расскажи стих
Model Output 2:
Раннее сияние золотых очей,
Свобода на белых губах,
Ноутбуки, виртуальные доски и мобильные телефоны,
Всегда рядом с ними.


Model Input 3:
когда собирать крыжовник?
Model Output 3:
Крыжовник — это древний способ сбора растительных топлива, который можно использовать для уменьшения загрязнения и поддержания окружающей среды. Вот несколько шагов, которые необходимо выполнить, чтобы собрать крыжовник:

1. Выберите место. Первым шагом в сборе крыжовника является выбор места для сбора. Это может быть сельскохозяйственная земля, лес или озера.

2. Соберите растения: собрать крыжовник можно с помощью различных растений, таких как зеленые растения, кустарники

Model Input 4:
Как быстро выучить новый язык?
Model Output 4:
Учение ново

1. Модель ответила - 8 планет уже хорошо
2. Стих - без обрезки получился, но нафантазировала конечно такое себе, публиковать нельзя такие стихи.
3. Касательно Крыжовника тоже фантазии полетели
4. Учения языков - вроде плюс минус вменяемо.

Что тут можно сказать:
- Для улучшения результатов, безусловно хорошо обучать на полном наборе данных, а не на той экономии, что у меня было.
- проработать можно с промтом
- дополнительные настройки температур, поставить например по ниже на 0.3, у нас стоит 0.7 - возможно будет меньше галюцинаций
- добавить количество эпох

В целом результат заметен между начальной генерацией модели и после нашей работы. Считаю, что поставленная задача в проекте - `Факты могут быть ошибочными, но язык ответа должен быть русским и должна сохраняться структура ответа.` - выполнена.
